# Design sequences with custom rewards
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/06_design_with_custom_rewards.ipynb)

Optimize a transparent charged-residue objective with length and entropy constraints. Define a reward, run a small GRPO experiment, reload the model, and measure generated samples.

This notebook runs independently. Select **Runtime → Change runtime type → GPU** in Colab.
First use downloads model weights. Training is a small workflow demonstration, not a converged design experiment. GPU memory requirements depend on the model, batch size, and length; a free Colab GPU is not guaranteed to fit RL-SAE.
The setup installs the `v1` release when IDiom is absent. If using an older installation,
upgrade to that release and restart the kernel. No adjacent helper files are required.

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "idiom[cookbook] @ git+https://github.com/rotskoff-group/idiom.git@v1"])
if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas>=2"])

import json
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from idiom import IDiom, IDiomSAE
from idiom.data.records import Record
from idiom.utils.notebook_helpers import (
    load_inputs, idr_sequence, isolated, check_context, summaries, write_fasta,
    save_run, sequence_metrics, nearest_reference, split_records, example_file,
)
print("Python:", sys.version.split()[0])
started = time.perf_counter()

## Settings
This notebook needs no input sequences. Edit the desired properties below. No tracking account
is required. Upload a scorer file to replace the example reward, or adapt its code in the next cell.
Use a fresh output directory; optionally point it into mounted Drive for persistent checkpoints.

In [ ]:
MODEL_ID = "jxliu2/idiom-20M"
DEVICE = "auto"
BATCH_SIZE = 1
SEED = 0
MAX_STEPS = 10
GROUP_SIZE = 2
LEARNING_RATE = 5e-6
TARGET_LENGTH = 50
MAX_NEW_TOKENS = 96
N = 8
RESUME_FROM = None
OUT_DIR = Path("reward_design_outputs")
TARGET_CHARGED_FRACTION = 0.3

In [ ]:
import gc
import torch
import lightning as L
from importlib.resources import files
from omegaconf import OmegaConf
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
L.seed_everything(SEED, workers=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if (OUT_DIR / "training").exists() and RESUME_FROM is None:
    raise ValueError("Use a fresh OUT_DIR or set RESUME_FROM to a training checkpoint.")
from idiom.utils.device import resolve_device
training_device = resolve_device(DEVICE)
precision = "bf16-mixed" if training_device.type == "cuda" and torch.cuda.is_bf16_supported() else "32-true"
print("CUDA:", torch.cuda.is_available(), "Training precision:", precision)

## Define a reward
A reward factory returns one finite score per sequence. Empty sequences need an explicit value.
This example measures the fraction of D/E/K/R residues and uses quadratic shaping around your target.
A score closer to zero is better for this shaped term. Length and entropy terms discourage shortcuts.
For a different objective, replace the scorer and its shaping settings together.

In [ ]:
reward_file = OUT_DIR / "custom_reward.py"
reward_file.write_text(
    'def charged_fraction():\n'
    '    def score(sequences):\n'
    '        return [sum(s.count(a) for a in "DEKR") / max(len(s), 1) for s in sequences]\n'
    '    return score\n'
)
reward_spec = dict(name=f"{reward_file.resolve()}:charged_fraction")
from idiom.train.grpo.reward.resolve import load_callable
scorer = load_callable(reward_spec["name"])()
assert scorer(["", "DEKR", "AAAA"]) == [0.0, 1.0, 0.0]

## Generate a baseline
Sample before training, then release the inference model to free memory.

In [ ]:
base = IDiom.from_pretrained(MODEL_ID, device=DEVICE)
sampling = dict(n=N, batch_size=BATCH_SIZE, seed=SEED, temperature=1.0,
                max_new_tokens=MAX_NEW_TOKENS)
baseline = base.generate_unprompted(**sampling)
write_fasta([Record(f"baseline_{i}", s, 0, len(s)) for i, s in enumerate(baseline) if s],
            OUT_DIR / "baseline.fasta")
model_context = base.model.cfg.max_seq_len
del base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Configure GRPO
The objective is the weighted sum of shaped rewards, with a KL penalty relative to the frozen
starting policy. Multiple completions per prompt provide within-group comparisons. These small
settings demonstrate execution; improvement is not guaranteed. Inspect raw component scores too.

In [ ]:
from idiom.train.grpo.train_grpo import build
from idiom.train.grpo.reward import build_reward
from idiom.train.grpo.data import collate_prompts
from torch.utils.data import DataLoader
cfg = OmegaConf.load(files("idiom") / "configs/grpo.yaml")
cfg.init_from = MODEL_ID
cfg.seed = SEED
cfg.prompts.n = max(16, MAX_STEPS * BATCH_SIZE)
cfg.prompts.batch_size = BATCH_SIZE
cfg.grpo.group_size = GROUP_SIZE
cfg.grpo.max_new_tokens = MAX_NEW_TOKENS
cfg.grpo.lr = LEARNING_RATE
cfg.grpo.track_disorder = False
cfg.grpo.log_samples_every = 0
cfg.trainer.max_steps = MAX_STEPS
cfg.out_dir = str(OUT_DIR.resolve())
cfg.resume_from = RESUME_FROM
cfg.device = str(training_device)
terms = [
    dict(label="length", weight=1.0, reward=dict(name="length"),
         shaping=dict(name="quadratic", target=TARGET_LENGTH, width=0.2)),
    dict(label="entropy", weight=1.0, reward=dict(name="entropy"),
         shaping=dict(name="quadratic", target=3.65, width=0.2)),
]

terms.append(dict(label="charged_fraction", weight=1.0, reward=reward_spec,
                  shaping=dict(name="quadratic", target=TARGET_CHARGED_FRACTION, width=0.2)))

cfg.reward.terms = terms
cfg.trainer = dict(max_steps=MAX_STEPS, accelerator="gpu" if training_device.type == "cuda" else "cpu",
                   devices=[training_device.index or 0] if training_device.type == "cuda" else 1,
                   precision=precision, gradient_clip_val=1.0, accumulate_grad_batches=1,
                   log_every_n_steps=1, limit_val_batches=2, num_sanity_val_steps=0,
                   enable_model_summary=False)
OmegaConf.save(cfg, OUT_DIR / "training_config.yaml")
lit, prompts = build(cfg)
loader = DataLoader(prompts, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_prompts)

In [ ]:
checkpoint = ModelCheckpoint(dirpath=OUT_DIR / "training/checkpoints", save_last=True,
                             save_top_k=0, every_n_train_steps=max(1, min(10, MAX_STEPS)))
trainer = L.Trainer(**OmegaConf.to_container(cfg.trainer, resolve=True),
                    logger=CSVLogger(OUT_DIR / "training", name="metrics"), callbacks=[checkpoint])
trainer.fit(lit, train_dataloaders=loader, ckpt_path=RESUME_FROM)

## Save, reload, and generate

In [ ]:
trainer.save_checkpoint(OUT_DIR / "training/checkpoints/last.ckpt")
release = OUT_DIR / "model"
IDiom(lit.model.cpu()).save_pretrained(release)
del trainer, lit
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
adapted_model = IDiom.from_pretrained(release, device=DEVICE)
adapted = adapted_model.generate_unprompted(**sampling)
write_fasta([Record(f"adapted_{i}", s, 0, len(s)) for i, s in enumerate(adapted) if s],
            OUT_DIR / "adapted.fasta")
del adapted_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Evaluate the complete objective and sequence quality

In [ ]:
comparison = pd.concat([sequence_metrics(baseline).assign(group="baseline"),
                        sequence_metrics(adapted).assign(group="adapted")], ignore_index=True)
comparison.to_csv(OUT_DIR / "candidates.csv", index=False)
display(comparison.groupby("group").agg(count=("sequence", "size"), mean_length=("length", "mean"),
                                        mean_entropy=("entropy", "mean"), duplicate_fraction=("duplicate", "mean")))
fig, axes = plt.subplots(1, 2, figsize=(8, 3), constrained_layout=True)
for group, rows in comparison.groupby("group"):
    axes[0].hist(rows.length, bins=10, alpha=0.5, label=group)
    axes[1].hist(rows.entropy, bins=10, alpha=0.5, label=group)
axes[0].set(xlabel="Length", ylabel="Count")
axes[1].set(xlabel="Composition entropy (bits)")
axes[0].legend()
fig.savefig(OUT_DIR / "comparison.png", dpi=160)
plt.show()

objective = build_reward(cfg.reward)
for group, sequences in (("baseline", baseline), ("adapted", adapted)):
    totals, details = objective(sequences, group_size=1)
    scores = pd.DataFrame(details)
    scores["total_reward"] = totals
    scores["sequence"] = sequences
    scores.to_csv(OUT_DIR / f"{group}_rewards.csv", index=False)
    print(group)
    display(scores)

In [ ]:
save_run(OUT_DIR, dict(model=MODEL_ID, seed=SEED, steps=MAX_STEPS, sampling=sampling,
                       reward=OmegaConf.to_container(cfg.reward, resolve=True), resume_from=RESUME_FROM),
         elapsed=time.perf_counter() - started)

## Save and continue
`model/` is a reloadable release; `training/checkpoints/last.ckpt` also preserves optimizer
state for resuming. `training/metrics/` contains CSV training logs. Keep the training config,
input audit, and run settings with generated sequences. These short runs demonstrate the workflow;
assess held-out data, diversity, and independent measurements before drawing design conclusions.
The download includes checkpoints and can be large; use Drive for long-running experiments.

In [ ]:
import shutil
archive = shutil.make_archive(str(OUT_DIR.resolve()), "zip", OUT_DIR)
print("Results:", OUT_DIR.resolve(), "\nDownload:", archive)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive)